# 05 Range Validation

Checks numeric domain ranges using streaming for telemetry.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import (
    RAW_DATA_PATH,
    EXPECTED_ENDPOINTS,
    TELEMETRY_ENDPOINTS,
    CRITICAL_COLUMNS,
    PRIMARY_KEYS,
    FOREIGN_KEYS,
    RANGE_RULES,
    THRESHOLDS,
    TECHNICAL_KEY_COLUMNS,
    DOMAIN_REVIEW_COLUMNS,
    STRUCTURAL_OPTIONAL_COLUMNS,
    ALLOWED_NULL_SCENARIOS,
    VALIDATION_SEVERITY,
    RANGE_SEVERITY_OVERRIDES,
)
from file_utils import build_file_inventory, endpoint_files, endpoint_files, iter_csv_endpoint
from validation_utils import endpoint_columns, null_profile, duplicate_count, range_violations

NOTEBOOK_NAME = "05_range_validation"
OUTPUT_TABLES = ROOT / "eda" / "bronze" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "bronze" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "bronze" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "bronze" / "insights"
CHECKPOINTS = ROOT / "eda" / "bronze" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(filename: str, title: str, summary: str, observations: list[str], issues: list[str], recommendations: list[str], next_steps: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += f"## Summary\n\n{summary}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n\n"
    content += "## Next Steps\n\n" + "\n".join(f"- {item}" for item in next_steps) + "\n"
    (INSIGHTS / filename).write_text(content, encoding="utf-8")

def p0_status_from_severity(df: pd.DataFrame) -> str:
    if df.empty or "severity" not in df.columns:
        return "PASS"
    blockers = df[df["severity"].eq("BLOCKER")]
    return "FAIL" if not blockers.empty else "PASS"

print("=" * 72)
print(f"BRONZE VALIDATION - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Raw data path: {RAW_DATA_PATH}")
print("=" * 72)


BRONZE VALIDATION - 05_range_validation
Start time: 2026-06-01 18:00:38.751704
Raw data path: D:\F1_WinRate_Predictor\data\raw


In [2]:
inventory_path = ROOT / "eda" / "bronze" / "outputs" / "tables" / "01_file_integrity" / "file_integrity.csv"
if inventory_path.exists():
    inventory = pd.read_csv(inventory_path)
else:
    inventory = build_file_inventory(RAW_DATA_PATH, EXPECTED_ENDPOINTS)

endpoint_column_cache = {}
for endpoint in EXPECTED_ENDPOINTS:
    observed = []
    endpoint_rows = inventory[inventory["endpoint"].eq(endpoint)]
    for value in endpoint_rows["columns"].dropna().astype(str):
        for column in value.split("|"):
            if column and column not in observed:
                observed.append(column)
    endpoint_column_cache[endpoint] = observed

records = []
chunk_size = int(THRESHOLDS.get("validation_chunk_size", 200000))
for endpoint in EXPECTED_ENDPOINTS:
    cols = endpoint_column_cache.get(endpoint, [])
    rules = {col: rule for col, rule in RANGE_RULES.items() if col in cols}
    if not rules:
        continue
    result = range_violations(RAW_DATA_PATH, endpoint, rules, endpoint in TELEMETRY_ENDPOINTS, chunksize=chunk_size)
    if not result.empty:
        result["endpoint_column"] = result["endpoint"] + "." + result["column"]
        result["severity"] = result["endpoint_column"].map(lambda key: RANGE_SEVERITY_OVERRIDES.get(key, "WARNING"))
        result["status"] = result.apply(lambda row: "PASS" if row["violations"] == 0 else ("FAIL" if row["severity"] == "BLOCKER" else "WARN"), axis=1)
        records.append(result)
range_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
range_df.to_csv(OUTPUT_TABLES / "range_violations.csv", index=False)
display(range_df.sort_values("violations", ascending=False).head(50))

,endpoint,column,checked_rows,violations,endpoint_column,severity,status,min_violation,max_violation,sample_values
25,race_control,driver_number,9736,8173,race_control.driver_number,INFO,WARN,NaN,NaN,NaN
26,race_control,lap_number,9736,3785,race_control.lap_number,INFO,WARN,0.000,0.000,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
9,laps,lap_duration,84072,1122,laps.lap_duration,WARNING,WARN,600.017,2127.296,"[661.864, 608.191, 980.349, 736.725, 655.227, ..."
13,weather,pressure,14902,480,weather.pressure,INFO,WARN,780.300,786.500,"[784.9, 784.8, 784.9, 784.8, 784.8, 784.8, 784..."
4,session_result,position,2740,0,session_result.position,WARNING,PASS,NaN,NaN,NaN
3,session_result,driver_number,2740,0,session_result.driver_number,WARNING,PASS,NaN,NaN,NaN
0,meetings,year,76,0,meetings.year,WARNING,PASS,NaN,NaN,NaN
6,session_result,points,1374,0,session_result.points,WARNING,PASS,NaN,NaN,NaN
5,session_result,duration,2740,0,session_result.duration,WARNING,PASS,NaN,NaN,NaN
8,laps,lap_number,84072,0,laps.lap_number,WARNING,PASS,NaN,NaN,NaN


In [3]:
fig = px.bar(range_df, x="endpoint", y="violations", color="column", barmode="group", title="Range Validation Violations")
fig.update_xaxes(tickangle=35)
fig.write_html(OUTPUT_CHARTS / "range_violations.html", include_plotlyjs="cdn")
try:
    fig.write_image(OUTPUT_CHARTS / "range_violations.png")
except Exception:
    pass
fig.show()

In [4]:
failed = range_df[(range_df["severity"] == "BLOCKER") & (range_df["violations"] > 0)] if not range_df.empty else pd.DataFrame()
warned = range_df[(range_df["violations"] > 0) & (range_df["severity"] != "BLOCKER")] if not range_df.empty else pd.DataFrame()
report = {"notebook": NOTEBOOK_NAME, "timestamp": datetime.now().isoformat(), "p0_status": "PASS" if failed.empty else "FAIL", "warning_count": int(len(warned)), "results": range_df.to_dict("records") if not range_df.empty else []}
write_report("range_validation", report)
write_insight(
    "05_range_insights.md",
    "Range Validation Insights",
    f"Validated numeric range rules across {range_df['endpoint'].nunique() if not range_df.empty else 0} endpoints.",
    [f"Columns checked: {range_df['column'].nunique() if not range_df.empty else 0}", f"Non-blocking range warnings: {len(warned)}"],
    [f"{row.endpoint}.{row.column}: {row.violations} BLOCKER violations" for row in failed.itertuples()],
    ["Use min_violation, max_violation, and sample_values columns to decide Silver mitigation.", "Treat weather pressure and lap-duration violations as contextual warnings unless confirmed impossible."],
    ["Run 06_temporal_checks.ipynb"],
)
print(report["p0_status"])

PASS
